# FER2013 VGG16 + GAP Paper Reproduction

Clean TensorFlow/Keras notebook for reproducing the paper-style fine-tuned VGG16 + GlobalAveragePooling2D FER2013 emotion classifier.

## 1. Imports

In [1]:
from pathlib import Path
from collections import Counter
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = Path.cwd()
IMAGE_SIZE = (48, 48)
INPUT_SHAPE = (48, 48, 3)
BATCH_SIZE = 32
EPOCHS = 50
CHECKPOINT_PATH = PROJECT_ROOT / "best_fer2013_vgg16_gap_paper_reproduction.keras"

CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}

# Common FER2013 folder-name variants are mapped to the canonical 7 paper classes.
CLASS_ALIASES = {
    "angry": "angry",
    "anger": "angry",
    "disgust": "disgust",
    "disgusted": "disgust",
    "fear": "fear",
    "fearful": "fear",
    "happy": "happy",
    "happiness": "happy",
    "neutral": "neutral",
    "sad": "sad",
    "sadness": "sad",
    "surprise": "surprise",
    "surprised": "surprise",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

print("Project root:", PROJECT_ROOT)
print("Classes:", CLASS_NAMES)

TensorFlow: 2.10.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Project root: c:\Users\nadee\Downloads\Emotion-Recognition
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## 2. Load Dataset

In [2]:
def first_existing_dir(root, names):
    """Return the first existing directory from a list of candidate names."""
    for name in names:
        candidate = root / name
        if candidate.is_dir():
            return candidate
    return None


def canonical_class_name(folder_name):
    """Map a dataset folder name to one of the 7 canonical FER2013 classes."""
    key = folder_name.strip().lower().replace(" ", "_")
    return CLASS_ALIASES.get(key)


def collect_image_paths(split_dir):
    """Collect image paths and integer labels from class subfolders."""
    image_paths = []
    labels = []
    skipped_folders = []

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        canonical_name = canonical_class_name(class_dir.name)
        if canonical_name is None:
            skipped_folders.append(class_dir.name)
            continue

        class_index = CLASS_TO_INDEX[canonical_name]
        files = sorted(
            path for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(files)
        labels.extend([class_index] * len(files))

    if skipped_folders:
        print(f"Skipped unknown folders in {split_dir.name}: {skipped_folders}")

    return np.array(image_paths, dtype=object), np.array(labels, dtype=np.int32)


train_dir = PROJECT_ROOT / "train"
validation_dir = first_existing_dir(PROJECT_ROOT, ["validation", "valid", "val", "public_test", "PublicTest", "public"])
private_test_dir = first_existing_dir(PROJECT_ROOT, ["private_test", "PrivateTest", "private"])
test_dir = PROJECT_ROOT / "test"

if not train_dir.is_dir():
    raise FileNotFoundError(f"Expected FER2013 training folder not found: {train_dir}")

train_paths, train_labels = collect_image_paths(train_dir)

if validation_dir is not None and private_test_dir is not None:
    val_paths, val_labels = collect_image_paths(validation_dir)
    test_paths, test_labels = collect_image_paths(private_test_dir)
    split_note = "Using explicit validation/public-test and private-test folders."
elif validation_dir is not None and test_dir.is_dir():
    val_paths, val_labels = collect_image_paths(validation_dir)
    test_paths, test_labels = collect_image_paths(test_dir)
    split_note = "Using explicit validation folder and test folder."
elif test_dir.is_dir():
    heldout_paths, heldout_labels = collect_image_paths(test_dir)
    label_counts = Counter(heldout_labels.tolist())
    stratify_labels = heldout_labels if label_counts and min(label_counts.values()) >= 2 else None
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        heldout_paths,
        heldout_labels,
        test_size=0.5,
        random_state=SEED,
        stratify=stratify_labels,
    )
    split_note = "No separate public/private folders found; split the FER2013 test folder 50/50 into validation and test."
else:
    raise FileNotFoundError(
        "Expected either validation/private_test folders or a test folder for FER2013 validation/test evaluation."
    )

print(split_note)
print("Train images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Test images:", len(test_paths))
print("Input shape:", INPUT_SHAPE)

No separate public/private folders found; split the FER2013 test folder 50/50 into validation and test.
Train images: 27469
Validation images: 3316
Test images: 3316
Input shape: (48, 48, 3)


## 3. Preprocessing

In [3]:
def load_and_preprocess_image(image_path, label):
    """Load a grayscale FER2013 image, normalize it to 0-1, convert to RGB, and one-hot encode the label."""
    image_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_image(image_bytes, channels=1, expand_animations=False)
    image.set_shape([None, None, 1])

    # FER2013 images are 48x48, but resize defensively in case a file differs.
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    # VGG16 with ImageNet weights requires 3 input channels.
    image = tf.image.grayscale_to_rgb(image)

    label = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label


def make_dataset(paths, labels, training=False):
    """Create a tf.data pipeline without balancing, class weighting, or augmentation."""
    dataset = tf.data.Dataset.from_tensor_slices(([str(path) for path in paths], labels))

    if training:
        dataset = dataset.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_ds = make_dataset(train_paths, train_labels, training=True)
train_eval_ds = make_dataset(train_paths, train_labels, training=False)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

sample_images, sample_labels = next(iter(train_ds))
print("Sample image batch shape:", sample_images.shape)
print("Sample label batch shape:", sample_labels.shape)
print("Pixel range:", float(tf.reduce_min(sample_images)), "to", float(tf.reduce_max(sample_images)))

Sample image batch shape: (32, 48, 48, 3)
Sample label batch shape: (32, 7)
Pixel range: 0.0 to 1.0


## 4. Class Distribution

In [4]:
def print_class_counts(split_name, labels):
    """Print counts for all 7 classes, including classes absent from a local folder."""
    counts = Counter(labels.tolist())
    print(f"{split_name} class counts:")
    for class_index, class_name in enumerate(CLASS_NAMES):
        print(f"  {class_name}: {counts.get(class_index, 0)}")
    print()


print_class_counts("Train", train_labels)
print_class_counts("Validation", val_labels)
print_class_counts("Test", test_labels)

Train class counts:
  angry: 3995
  disgust: 0
  fear: 4097
  happy: 6411
  neutral: 4965
  sad: 4830
  surprise: 3171

Validation class counts:
  angry: 479
  disgust: 0
  fear: 512
  happy: 670
  neutral: 617
  sad: 623
  surprise: 415

Test class counts:
  angry: 479
  disgust: 0
  fear: 512
  happy: 669
  neutral: 616
  sad: 624
  surprise: 416



## 5. Build VGG16 + GAP Model

In [5]:
base_model = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE,
)

# Fine-tune the full VGG16 base model, matching the paper's non-frozen setup.
base_model.trainable = True

model = tf.keras.Sequential(
    [
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="fer2013_vgg16_gap_paper_reproduction",
)

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print("VGG16 base trainable:", base_model.trainable)
print("Trainable VGG16 layers:", sum(layer.trainable for layer in base_model.layers), "/", len(base_model.layers))

Model: "fer2013_vgg16_gap_paper_reproduction"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 global_average_pooling2d (G  (None, 512)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 7)                 3591      
                                                                 
Total params: 14,718,279
Trainable params: 14,718,279
Non-trainable params: 0
_________________________________________________________________
VGG16 base trainable: True
Trainable VGG16 layers: 19 / 19


## 6. Train Model

In [6]:
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True,
    verbose=1,
)

checkpoint = ModelCheckpoint(
    filepath=str(CHECKPOINT_PATH),
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint],
)

Epoch 1/50
859/859 [==============================] - ETA: 0s - loss: 1.3167 - accuracy: 0.4789
Epoch 1: val_accuracy improved from -inf to 0.51267, saving model to c:\Users\nadee\Downloads\Emotion-Recognition\best_fer2013_vgg16_gap_paper_reproduction.keras
859/859 [==============================] - 27s 30ms/step - loss: 1.3167 - accuracy: 0.4789 - val_loss: 1.2024 - val_accuracy: 0.5127
Epoch 2/50
857/859 [============================>.] - ETA: 0s - loss: 1.0818 - accuracy: 0.5795
Epoch 2: val_accuracy improved from 0.51267 to 0.56906, saving model to c:\Users\nadee\Downloads\Emotion-Recognition\best_fer2013_vgg16_gap_paper_reproduction.keras
859/859 [==============================] - 26s 30ms/step - loss: 1.0821 - accuracy: 0.5793 - val_loss: 1.1023 - val_accuracy: 0.5691
Epoch 3/50
857/859 [============================>.] - ETA: 0s - loss: 0.9817 - accuracy: 0.6258
Epoch 3: val_accuracy improved from 0.56906 to 0.58414, saving model to c:\Users\nadee\Downloads\Emotion-Recognition\be

NotFoundError: Graph execution error:

2 root error(s) found.
  (0) NOT_FOUND:  NewRandomAccessFile failed to Create/Open: c:\Users\nadee\Downloads\Emotion-Recognition\train\neutral\im615.png : The system cannot find the path specified.
; No such process
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]]
	 [[categorical_crossentropy/softmax_cross_entropy_with_logits/Shape_2/_10]]
  (1) NOT_FOUND:  NewRandomAccessFile failed to Create/Open: c:\Users\nadee\Downloads\Emotion-Recognition\train\neutral\im615.png : The system cannot find the path specified.
; No such process
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_train_function_1923]

## 7. Evaluate Model

In [ ]:
if CHECKPOINT_PATH.exists():
    print("Loading best checkpoint:", CHECKPOINT_PATH)
    model = tf.keras.models.load_model(CHECKPOINT_PATH)

train_loss, train_accuracy = model.evaluate(train_eval_ds, verbose=0)
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Test loss: {test_loss:.4f}")

test_probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(test_probabilities, axis=1)
y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds], axis=0)

print("Classification report - test set:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
print("Confusion matrix - test set:")
print(cm)

print("Paper reported accuracy: 69.40%")
print(f"This run test accuracy: {test_accuracy * 100:.2f}%")

## 8. Plots and Confusion Matrix

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training Accuracy vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 7))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix - Test Set")
plt.colorbar()
tick_marks = np.arange(NUM_CLASSES)
plt.xticks(tick_marks, CLASS_NAMES, rotation=45, ha="right")
plt.yticks(tick_marks, CLASS_NAMES)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

threshold = cm.max() / 2.0 if cm.size else 0
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        color = "white" if cm[row, col] > threshold else "black"
        plt.text(col, row, cm[row, col], ha="center", va="center", color=color)

plt.tight_layout()
plt.show()